<a href="https://colab.research.google.com/github/giuliobarde/web_data_mining_project/blob/main/WebDataMiningMiniProject3Group1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --upgrade pip
!pip install \
  pinecone-client \
  boto3 \
  pandas \
  langchain \
  langchain-openai \
  langchain-pinecone \
  langchain_community \
  langgraph-prebuilt

In [30]:
import os
from getpass import getpass

import boto3
import pandas as pd

from pinecone import Pinecone as PineconeClient

from langchain_openai import ChatOpenAI
from langchain_pinecone import Pinecone as LC_Pinecone
from langchain.chains import RetrievalQA
from langchain.tools import Tool

from langgraph.prebuilt import create_react_agent


In [15]:
PINECONE_API_KEY = getpass("Enter your Pinecone Key: ")
PINECONE_ENV = "us-east-1"
OPENAI_API_KEY = getpass("Enter your OpenAI key: ")
INDEX_NAME = "cus635"
NAMESPACE = "Team_1"

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY


Enter your Pinecone Key: ··········
Enter your OpenAI key: ··········


In [31]:
pc = PineconeClient(api_key=PINECONE_API_KEY, environment=PINECONE_ENV)

if INDEX_NAME not in pc.list_indexes().names():
    raise ValueError(f"Index '{INDEX_NAME}' not found.")

pine_index = pc.Index(INDEX_NAME)


In [32]:
from langchain_openai import OpenAIEmbeddings

embeddings  = OpenAIEmbeddings()
vectorstore = LC_Pinecone(
    index=pine_index,
    embedding=embeddings,
    text_key="text",
    namespace=NAMESPACE
)


In [33]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
qa_chain  = RetrievalQA.from_chain_type(
    llm=ChatOpenAI(temperature=0, model="gpt-4o-mini"),
    chain_type="stuff",
    retriever=retriever
)


In [38]:
def search_news(query: str) -> str:
    result = qa_chain.invoke({"query": query})
    return result["result"]

news_tool = Tool.from_function(
    func=search_news,
    name="search_news",
    description="Ask questions about the ingested news articles."
)


In [41]:
llm = ChatOpenAI(temperature=0.2, model="gpt-4.1-nano-2025-04-14")
agent = create_react_agent(
    llm,
    tools=[news_tool]
)


In [44]:
resp = agent.invoke({
    "messages": [{"role": "user", "content": "What are the latest investment trends?"}]
})

# Grab the assistant’s reply (the last message in the list)
print(resp["messages"][-1].content)


It appears there is a technical issue preventing me from accessing the latest news on investment trends. However, I can provide a general overview of recent investment trends based on available knowledge up to October 2023. Would you like me to do that?
